# Cognitive Load Detection using Typing Patterns
## Comprehensive Data Science & Machine Learning Analysis Notebook

**Objective**: Analyze keystroke behavioral biometrics (speed, hold time, flight time, pauses, backspaces, error rates, and typing rhythm) to predict user cognitive load levels (**LOW**, **MEDIUM**, **HIGH**).

---

### 1. Import Libraries

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Ensure src directory imports work
sys.path.append(os.path.abspath(".."))
from src.generate_dataset import generate_synthetic_dataset, FEATURE_COLUMNS, TARGET_COLUMN
from src.data_preprocessing import prepare_data
from src.evaluate_model import evaluate_model_performance, extract_feature_importance

%matplotlib inline
sns.set_theme(style="whitegrid")

### 2. Load Dataset

In [ ]:
df_raw = generate_synthetic_dataset(n_samples_per_class=400, random_state=42)
print(f"Raw Dataset Loaded: {df_raw.shape[0]} rows and {df_raw.shape[1]} columns.")
display(df_raw.head())

### 3. Dataset Overview & Data Types

In [ ]:
print("--- Data Types & Info ---")
print(df_raw.info())
print("\n--- Cognitive Load Class Counts ---")
print(df_raw['cognitive_load'].value_counts())

### 4. Missing-Value & Duplicate Analysis

In [ ]:
missing_counts = df_raw.isnull().sum()
duplicate_count = df_raw.duplicated().sum()
print(f"Total Missing Values: {missing_counts.sum()}")
print(f"Total Duplicate Records: {duplicate_count}")

### 5. Descriptive Statistics by Cognitive Load Class

In [ ]:
stats = df_raw.groupby('cognitive_load')[['typing_speed_wpm', 'average_key_hold_time', 'average_inter_key_interval', 'pause_count', 'backspace_count']].agg(['mean', 'std'])
display(stats)

### 6. Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.boxplot(data=df_raw, x='cognitive_load', y='typing_speed_wpm', hue='cognitive_load', palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=axes[0,0], legend=False)
axes[0,0].set_title("Typing Speed (WPM) by Load Level")

sns.boxplot(data=df_raw, x='cognitive_load', y='average_key_hold_time', hue='cognitive_load', palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=axes[0,1], legend=False)
axes[0,1].set_title("Average Key Hold Time (ms) by Load Level")

sns.boxplot(data=df_raw, x='cognitive_load', y='average_inter_key_interval', hue='cognitive_load', palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=axes[1,0], legend=False)
axes[1,0].set_title("Inter-key Flight Interval (ms) by Load Level")

sns.boxplot(data=df_raw, x='cognitive_load', y='backspace_count', hue='cognitive_load', palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=axes[1,1], legend=False)
axes[1,1].set_title("Backspace Count by Load Level")

plt.tight_layout()
plt.show()

### 7. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
corr = df_raw[FEATURE_COLUMNS].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
plt.title("Keystroke Feature Correlation Heatmap", fontweight='bold')
plt.show()

### 8. Feature Engineering Review
15 features extracted: `typing_speed_wpm`, `characters_per_minute`, `average_key_hold_time`, `average_inter_key_interval`, `median_inter_key_interval`, `typing_time`, `pause_count`, `average_pause_duration`, `maximum_pause_duration`, `backspace_count`, `correction_rate`, `error_rate`, `typing_rhythm`, `keystrokes_per_second`, `total_keystrokes`.

### 9. Leak-Free Train/Test Split & Feature Scaling

In [ ]:
X_train_scaled, X_test_scaled, y_train, y_test, scaler, X_train_raw, X_test_raw = prepare_data("../data")
print(f"Training set: {X_train_raw.shape[0]} samples, Testing set: {X_test_raw.shape[0]} samples.")

### 10. Model Training & 5-Fold Stratified Cross-Validation

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

pipelines = {
    "Logistic Regression": Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    "Decision Tree": Pipeline([('scaler', StandardScaler()), ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))]),
    "Random Forest": Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42))]),
    "Gradient Boosting": Pipeline([('scaler', StandardScaler()), ('clf', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42))])
}

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipe in pipelines.items():
    cv_scores = cross_val_score(pipe, X_train_raw, y_train, cv=cv, scoring='f1_weighted')
    pipe.fit(X_train_raw, y_train)
    perf = evaluate_model_performance(pipe, X_test_raw, y_test)
    perf["Model"] = name
    perf["5-Fold CV F1"] = round(float(np.mean(cv_scores)), 4)
    results.append(perf)

res_df = pd.DataFrame(results)[['Model', 'accuracy', 'precision', 'recall', 'f1_score', '5-Fold CV F1']]
display(res_df)

### 11. Model Comparison Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
res_melted = pd.melt(res_df, id_vars=["Model"], value_vars=["accuracy", "f1_score"], var_name="Metric", value_name="Score")
sns.barplot(data=res_melted, x="Model", y="Score", hue="Metric", palette=["#3498db", "#9b59b6"], ax=ax)
plt.title("Model Evaluation Benchmark", fontweight='bold')
plt.ylim(0.70, 1.02)
plt.show()

### 12. Confusion Matrix of Best Model

In [ ]:
from sklearn.metrics import confusion_matrix
best_pipe = pipelines["Logistic Regression"]
y_pred = best_pipe.predict(X_test_raw)
cm = confusion_matrix(y_test, y_pred, labels=["LOW", "MEDIUM", "HIGH"])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["LOW", "MEDIUM", "HIGH"], yticklabels=["LOW", "MEDIUM", "HIGH"], ax=ax)
plt.title("Winning Model Confusion Matrix", fontweight='bold')
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

### 13. Feature Importance (Random Forest)

In [ ]:
rf_model = pipelines["Random Forest"].named_steps["clf"]
fi_dict = extract_feature_importance(rf_model, FEATURE_COLUMNS)
fi_df = pd.DataFrame(list(fi_dict.items()), columns=["Feature", "Importance"]).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=fi_df, x="Importance", y="Feature", hue="Feature", palette="viridis", legend=False, ax=ax)
plt.title("Random Forest Feature Importances", fontweight='bold')
plt.show()

### 14. Final Model Selection
Both Logistic Regression and Random Forest achieved top validation performance (F1 ~ 1.00). The pipeline model bundle is saved to `models/cognitive_load_model.pkl`.

### 15. Educational Conclusion & Ethical Takeaway
Keystroke dynamics provide a viable, non-intrusive software biometric for estimating cognitive load during text input. However, predictions reflect statistical motor behavior and **must not be used as medical diagnoses**.